In [1]:
# Q7 - Previous campaign

#Question: do customers who were contacted in an earlier campaign respond differently?

# `previous_campaign_outcome` is the outcome of the previous campaign ("nonexistent" means no earlier contact, decision 9).
# `contacts_before_campaign` counts earlier contacts. `days_since_previous_contact` is recorded for only 1,515 customers (decision 2).
# The data has no customer ID, so earlier records cannot be linked to individual customers. Each group is compared with the
# overall yes rate (the Q1 baseline) and shown with its customer count. Groups under 100 customers are flagged and not interpreted.

In [2]:
import pandas as pd

CLEAN_PATH = "../../data/bank_marketing_clean.parquet"
clean_df = pd.read_parquet(CLEAN_PATH)

clean_df.shape

(41176, 22)

In [3]:
# Q7, step 1: yes rate by previous campaign outcome and by number of earlier contacts
subscribed = clean_df["subscribed_term_deposit"] == "yes"
baseline_rate = subscribed.mean() * 100

earlier_contacts = clean_df["contacts_before_campaign"]
earlier_band = pd.cut(earlier_contacts, bins=[0, 1, 2, 3, 100],
                      labels=["0", "1", "2", "3 or more"], right=False)
contacted_before = clean_df["previously_contacted"].map({False: "never contacted before", True: "contacted before"})

groupings = {
    "previous_campaign_outcome": (clean_df["previous_campaign_outcome"], ["nonexistent", "failure", "success"]),
    "earlier_contacts": (earlier_band, ["0", "1", "2", "3 or more"]),
    "contacted_before": (contacted_before, ["never contacted before", "contacted before"]),
}

summaries = []
for name, (values, order) in groupings.items():
    summary = (
        pd.DataFrame({"value": values, "subscribed": subscribed})
        .groupby("value", observed=True)["subscribed"]
        .agg(customers="count", subscribers="sum")
        .reindex(order)
    )
    summary["yes_rate_pct"] = (summary["subscribers"] / summary["customers"] * 100).round(2)
    summary["vs_baseline_points"] = (summary["yes_rate_pct"] - baseline_rate).round(2)
    summary["share_of_customers_pct"] = (summary["customers"] / len(clean_df) * 100).round(2)
    summary["share_of_subscribers_pct"] = (summary["subscribers"] / subscribed.sum() * 100).round(2)
    summary["small_group"] = summary["customers"] < 100
    summaries.append(summary)

    print(f"{name}: customers {summary['customers'].sum()} (should be {len(clean_df)}), "
          f"subscribers {summary['subscribers'].sum()} (should be {subscribed.sum()})")

print(f"Baseline yes rate: {baseline_rate:.2f}%")
print()
pd.concat(summaries, keys=list(groupings), names=["grouping", "value"])

previous_campaign_outcome: customers 41176 (should be 41176), subscribers 4639 (should be 4639)
earlier_contacts: customers 41176 (should be 41176), subscribers 4639 (should be 4639)
contacted_before: customers 41176 (should be 41176), subscribers 4639 (should be 4639)
Baseline yes rate: 11.27%



customers  subscribers  \
grouping                  value                                            
previous_campaign_outcome nonexistent                 35551         3140   
                          failure                      4252          605   
                          success                      1373          894   
earlier_contacts          0                           35551         3140   
                          1                            4561          967   
                          2                             754          350   
                          3 or more                     310          182   
contacted_before          never contacted before      35551         3140   
                          contacted before             5625         1499   

                                                  yes_rate_pct  \
grouping                  value                                  
previous_campaign_outcome nonexistent                     8.83   
                          failure                        14.23   
                          success                        65.11   
earlier_contacts          0                               8.83   
                          1                              21.20   
                          2                              46.42   
                          3 or more                      58.71   
contacted_before          never contacted before          8.83   
                          contacted before               26.65   

                                                  vs_baseline_points  \
grouping                  value                                        
previous_campaign_outcome nonexistent                          -2.44   
                          failure                               2.96   
                          success                              53.84   
earlier_contacts          0                                    -2.44   
                          1                                     9.93   
                          2                                    35.15   
                          3 or more                            47.44   
contacted_before          never contacted before               -2.44   
                          contacted before                     15.38   

                                                  share_of_customers_pct  \
grouping                  value                                            
previous_campaign_outcome nonexistent                              86.34   
                          failure                                  10.33   
                          success                                   3.33   
earlier_contacts          0                                        86.34   
                          1                                        11.08   
                          2                                         1.83   
                          3 or more                                 0.75   
contacted_before          never contacted before                   86.34   
                          contacted before                         13.66   

                                                  share_of_subscribers_pct  \
grouping                  value                                              
previous_campaign_outcome nonexistent                                67.69   
                          failure                                    13.04   
                          success                                    19.27   
earlier_contacts          0                                          67.69   
                          1                                          20.85   
                          2                                           7.54   
                          3 or more                                   3.92   
contacted_before          never contacted before                     67.69   
                          contacted before                           32.31  

In [4]:
# Q7, step 2: previous campaign outcome against month
month_order = ["mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"]
outcome_order = ["nonexistent", "failure", "success"]

frame = pd.DataFrame({
    "month": clean_df["contact_month"],
    "outcome": clean_df["previous_campaign_outcome"],
    "subscribed": subscribed,
})

grouped = frame.groupby(["month", "outcome"])["subscribed"].agg(customers="count", subscribers="sum")
customers = (grouped["customers"].unstack("outcome")
             .reindex(index=month_order, columns=outcome_order).fillna(0).astype(int))
subscribers = (grouped["subscribers"].unstack("outcome")
               .reindex(index=month_order, columns=outcome_order).fillna(0).astype(int))

# Reconcile against the earlier tables
expected_month_totals = pd.Series({
    "mar": 546, "apr": 2631, "may": 13767, "jun": 5318, "jul": 7169,
    "aug": 6176, "sep": 570, "oct": 717, "nov": 4100, "dec": 182,
})
print(f"Customers: {customers.values.sum()} (should be {len(clean_df)}), "
      f"subscribers: {subscribers.values.sum()} (should be {subscribed.sum()})")
print("Customers by outcome:", customers.sum().to_dict(),
      "(should be nonexistent 35551, failure 4252, success 1373)")
print("Subscribers by outcome:", subscribers.sum().to_dict(),
      "(should be nonexistent 3140, failure 605, success 894)")
print("Month totals match Question 5:", bool((customers.sum(axis=1) == expected_month_totals).all()))
print()

customers.loc["all months"] = customers.sum()
subscribers.loc["all months"] = subscribers.sum()

mix_table = pd.DataFrame({
    "never_contacted_customers": customers["nonexistent"],
    "failure_customers": customers["failure"],
    "success_customers": customers["success"],
    "total_customers": customers.sum(axis=1),
    "pct_contacted_before": ((customers["failure"] + customers["success"]) / customers.sum(axis=1) * 100).round(1),
    "pct_earlier_success": (customers["success"] / customers.sum(axis=1) * 100).round(1),
})

rates = (subscribers / customers * 100).where(customers >= 100).round(2)
rate_table = pd.DataFrame({
    "never_contacted_yes_rate_pct": rates["nonexistent"],
    "failure_yes_rate_pct": rates["failure"],
    "success_yes_rate_pct": rates["success"],
    "all_customers_yes_rate_pct": (subscribers.sum(axis=1) / customers.sum(axis=1) * 100).round(2),
    "pct_of_subscriptions_from_earlier_success": (subscribers["success"] / subscribers.sum(axis=1) * 100).round(1),
})

print("Customers by month and previous outcome:")
print(mix_table)
print()
print("Yes rate (%) by month and previous outcome (blank where under 100 customers):")
rate_table

Customers: 41176 (should be 41176), subscribers: 4639 (should be 4639)
Customers by outcome: {'nonexistent': 35551, 'failure': 4252, 'success': 1373} (should be nonexistent 35551, failure 4252, success 1373)
Subscribers by outcome: {'nonexistent': 3140, 'failure': 605, 'success': 894} (should be nonexistent 3140, failure 605, success 894)
Month totals match Question 5: True

Customers by month and previous outcome:
            never_contacted_customers  failure_customers  success_customers  \
month                                                                         
mar                               367                 98                 81   
apr                              1873                647                111   
may                             11758               1779                230   
jun                              5003                171                144   
jul                              6941                120                108   
aug                         

,never_contacted_yes_rate_pct,failure_yes_rate_pct,success_yes_rate_pct,all_customers_yes_rate_pct,pct_of_subscriptions_from_earlier_success
month,,,,,
mar,47.41,NaN,NaN,50.55,23.2
apr,21.20,11.28,62.16,20.49,12.8
may,5.47,7.31,49.13,6.44,12.8
jun,8.20,31.58,65.97,10.51,17.0
jul,7.45,40.83,75.93,9.04,12.7
aug,7.71,31.10,65.85,10.61,20.6
sep,35.44,29.20,77.70,44.91,44.9
oct,39.23,33.93,70.59,43.93,30.5
nov,8.04,8.58,57.58,10.15,22.8


In [5]:
# Q7, step 3: number of earlier contacts within each previous outcome
outcome_order = ["failure", "success"]
band_order = ["1", "2", "3 or more"]

earlier_df = pd.DataFrame({
    "outcome": clean_df["previous_campaign_outcome"],
    "earlier_contacts": earlier_band,
    "subscribed": subscribed,
})
earlier_df = earlier_df[earlier_df["outcome"] != "nonexistent"]

grouped = (earlier_df.groupby(["outcome", "earlier_contacts"], observed=True)["subscribed"]
           .agg(customers="count", subscribers="sum"))

# Reconcile against the earlier tables
print(f"Customers: {grouped['customers'].sum()} (should be 5625), "
      f"subscribers: {grouped['subscribers'].sum()} (should be 1499)")
print("By outcome (customers / subscribers):",
      grouped.groupby("outcome").sum().loc[outcome_order].values.tolist(),
      "(should be [[4252, 605], [1373, 894]])")
print("By earlier contacts (customers / subscribers):",
      grouped.groupby("earlier_contacts", observed=True).sum().loc[band_order].values.tolist(),
      "(should be [[4561, 967], [754, 350], [310, 182]])")
print()

rows = []
for outcome in outcome_order:
    for band in band_order + ["all"]:
        if band == "all":
            customers = grouped.loc[outcome, "customers"].sum()
            subscribers = grouped.loc[outcome, "subscribers"].sum()
        elif (outcome, band) in grouped.index:
            customers = grouped.loc[(outcome, band), "customers"]
            subscribers = grouped.loc[(outcome, band), "subscribers"]
        else:
            customers, subscribers = 0, 0
        rows.append({
            "outcome": outcome,
            "earlier_contacts": band,
            "customers": customers,
            "subscribers": subscribers,
            "yes_rate_pct": round(subscribers / customers * 100, 2) if customers >= 100 else None,
            "small_group": customers < 100,
        })

result = pd.DataFrame(rows).set_index(["outcome", "earlier_contacts"])
print("Yes rate by previous outcome and number of earlier contacts (blank where under 100 customers):")
print(result)
print()

counts = earlier_df.groupby(["earlier_contacts", "outcome"], observed=True).size().unstack("outcome")
share_table = counts.reindex(index=band_order, columns=outcome_order).fillna(0).astype(int)
share_table["total_customers"] = share_table.sum(axis=1)
share_table["pct_earlier_success"] = (share_table["success"] / share_table["total_customers"] * 100).round(1)
print("Share of each earlier-contact group that had an earlier success:")
share_table

Customers: 5625 (should be 5625), subscribers: 1499 (should be 1499)
By outcome (customers / subscribers): [[4252, 605], [1373, 894]] (should be [[4252, 605], [1373, 894]])
By earlier contacts (customers / subscribers): [[4561, 967], [754, 350], [310, 182]] (should be [[4561, 967], [754, 350], [310, 182]])

Yes rate by previous outcome and number of earlier contacts (blank where under 100 customers):
                          customers  subscribers  yes_rate_pct  small_group
outcome earlier_contacts                                                   
failure 1                      3696          451         12.20        False
        2                       434          112         25.81        False
        3 or more               122           42         34.43        False
        all                    4252          605         14.23        False
success 1                       865          516         59.65        False
        2                       320          238         74.38  

outcome,failure,success,total_customers,pct_earlier_success
earlier_contacts,,,,
1,3696,865,4561,19.0
2,434,320,754,42.4
3 or more,122,188,310,60.6


In [6]:
# Q7, step 4: days since the previous contact (customers with a recorded gap only)
days = clean_df["days_since_previous_contact"]
has_gap = days.notna()

gap_df = pd.DataFrame({
    "days": days[has_gap].astype(int),
    "outcome": clean_df.loc[has_gap, "previous_campaign_outcome"],
    "subscribed": subscribed[has_gap],
})

gap_edges = [0, 3, 5, 7, 8, 15, 100]
gap_labels = ["0-2", "3-4", "5-6", "7", "8-14", "15 and over"]
gap_df["gap_band"] = pd.cut(gap_df["days"], bins=gap_edges, labels=gap_labels, right=False)

print(f"Customers with a recorded gap: {len(gap_df)} (should be 1515)")
print("By outcome (customers):", gap_df["outcome"].value_counts().to_dict(),
      "(should be success 1373, failure 142)")
print(f"Earlier successes who subscribed: {gap_df[gap_df['outcome'] == 'success']['subscribed'].sum()} (should be 894)")
print(f"Days range: {gap_df['days'].min()} to {gap_df['days'].max()}, median {gap_df['days'].median()}")
print()

overall = (gap_df.groupby("gap_band", observed=True)["subscribed"]
           .agg(customers="count", subscribers="sum").reindex(gap_labels))
overall["yes_rate_pct"] = (overall["subscribers"] / overall["customers"] * 100).round(2)
overall["share_of_customers_pct"] = (overall["customers"] / len(gap_df) * 100).round(1)
overall["small_group"] = overall["customers"] < 100
print(f"Customers in bands: {overall['customers'].sum()} (should be 1515)")
print()
print("Yes rate by days since the previous contact (all customers with a recorded gap):")
print(overall)
print()

by_outcome = (gap_df.groupby(["outcome", "gap_band"], observed=True)["subscribed"]
              .agg(customers="count", subscribers="sum"))
customers_pivot = (by_outcome["customers"].unstack("outcome")
                   .reindex(index=gap_labels, columns=["success", "failure"]).fillna(0).astype(int))
subscribers_pivot = (by_outcome["subscribers"].unstack("outcome")
                     .reindex(index=gap_labels, columns=["success", "failure"]).fillna(0).astype(int))
rates = (subscribers_pivot / customers_pivot * 100).where(customers_pivot >= 100).round(2)
by_outcome_table = pd.DataFrame({
    "success_customers": customers_pivot["success"],
    "success_yes_rate_pct": rates["success"],
    "failure_customers": customers_pivot["failure"],
    "failure_yes_rate_pct": rates["failure"],
})
print("By previous outcome (yes rates blank where under 100 customers):")
by_outcome_table

Customers with a recorded gap: 1515 (should be 1515)
By outcome (customers): {'success': 1373, 'failure': 142} (should be success 1373, failure 142)
Earlier successes who subscribed: 894 (should be 894)
Days range: 0 to 27, median 6.0

Customers in bands: 1515 (should be 1515)

Yes rate by days since the previous contact (all customers with a recorded gap):
             customers  subscribers  yes_rate_pct  share_of_customers_pct  \
gap_band                                                                    
0-2                102           55         53.92                     6.7   
3-4                557          361         64.81                    36.8   
5-6                458          318         69.43                    30.2   
7                   60           40         66.67                     4.0   
8-14               276          157         56.88                    18.2   
15 and over         62           36         58.06                     4.1   

             small_grou

,success_customers,success_yes_rate_pct,failure_customers,failure_yes_rate_pct
gap_band,,,,
0-2,102,53.92,0,NaN
3-4,551,65.15,6,NaN
5-6,429,70.40,29,NaN
7,45,NaN,15,NaN
8-14,210,58.10,66,NaN
15 and over,36,NaN,26,NaN
